## Business Objective

We want to measure whether showing users Ad B (exposed) increases engagement compared to Ad A (control).

What “engagement” means here

Engagement = user answers YES to the questionnaire.
So this is a binary outcome (YES=1, NO=0) → perfect for a proportion test.

### Experiment Setup


Control (A): dummy ad

Exposed (B): new smart ad

Key Metric (Primary Metric)

Conversion Rate / Success Rate

##  Hypotheses

### Definition

- **Null Hypothesis (H₀):** There is no real difference between the two ads.  
  Any observed difference is due to random variation.

- **Alternative Hypothesis (H₁):** There is a real difference between the two ads.

---

### Hypothesis Formulation (Two-Sided Test)

\[
H_0: p_A = p_B
\]

\[
H_1: p_A \ne p_B
\]

Where:

- \(p_A\) = conversion rate of **Ad A (control group)**
- \(p_B\) = conversion rate of **Ad B (exposed group)**

---

### Significance Level

\[
alpha = 0.05
\]

This means we allow a **5% probability of Type I error** (false positive).

In other words, we accept a 5% chance of incorrectly rejecting the null hypothesis when it is actually true.


### Load Data

In [4]:
import numpy as np
import pandas as pd
from scipy.stats import norm

df = pd.read_csv("../data/AdSmartABdata - AdSmartABdata.csv")
df.head()

,auction_id,experiment,date,hour,device_make,platform_os,browser,yes,no
0,0008ef63-77a7-448b-bd1e-075f42c55e39,exposed,2020-07-10,8,Generic Smartphone,6,Chrome Mobile,0,0
1,000eabc5-17ce-4137-8efe-44734d914446,exposed,2020-07-07,10,Generic Smartphone,6,Chrome Mobile,0,0
2,0016d14a-ae18-4a02-a204-6ba53b52f2ed,exposed,2020-07-05,2,E5823,6,Chrome Mobile WebView,0,1
3,00187412-2932-4542-a8ef-3633901c98d9,control,2020-07-03,15,Samsung SM-A705FN,6,Facebook,0,0
4,001a7785-d3fe-4e11-a344-c8735acacc2c,control,2020-07-03,15,Generic Smartphone,6,Chrome Mobile,0,0


### Preprocessing: Remove Non-Responses
Why?

Rows with yes=0 and no=0 mean the user did not answer.
If our “engagement” metric is answering YES/NO, those are not informative for this analysis.

In [5]:
non_response = df[(df["yes"]==0) & (df["no"]==0)]
df2 = df.drop(non_response.index)

df.shape, df2.shape

((8077, 9), (1243, 9))

### Count Users + YES per Group

In [7]:
df2.head(2)

,auction_id,experiment,date,hour,device_make,platform_os,browser,yes,no
2,0016d14a-ae18-4a02-a204-6ba53b52f2ed,exposed,2020-07-05,2,E5823,6,Chrome Mobile WebView,0,1
16,008aafdf-deef-4482-8fec-d98e3da054da,exposed,2020-07-04,16,Generic Smartphone,6,Chrome Mobile,1,0


#group sizes 

n = df2["experiment"].value_counts()
n_control = n["control"]
n_exposed = n["exposed"]

n, n_control, n_exposed

In [10]:
#Yes counts 

yes_counts = df2.groupby("experiment")["yes"].sum()
x_control = yes_counts["control"]
x_exposed = yes_counts["exposed"]

yes_counts, x_control, x_exposed

(experiment
 control    264
 exposed    308
 Name: yes, dtype: int64,
 264,
 308)

In [11]:
n_control, n_exposed, x_control, x_exposed

(586, 657, 264, 308)

In [ ]:
n_control, n_exposed, x_control, x_exposed

### Compute Conversion Rates

In [13]:
p_control = x_control / n_control
p_exposed = x_exposed / n_exposed
diff = p_exposed - p_control

p_control, p_exposed, diff

(0.45051194539249145, 0.4687975646879756, 0.018285619295484168)

### Interpretation

If diff > 0: exposed performs better
    Here the differe is 0.018 

But we still need to test if it’s statistically significant

### What is a Z-Test 

A Z-test checks whether the difference between two conversion rates is large compared to expected random variation

## 9) Pooled Proportion 

### Definition

Under the null hypothesis that the conversion rates are equal:

\[
H_0 : p_A = p_B
\]

we assume that both groups share the **same true conversion rate**.

To estimate this common conversion rate, we use the **pooled proportion**.

---

### Formula
p hat = (x_A + x_B) / (n_A + n_B)


Where:

- \(x_A\) = number of conversions in group A  
- \(x_B\) = number of conversions in group B  
- \(n_A\) = total observations in group A  
- \(n_B\) = total observations in group B  

---

### Interpretation

The pooled proportion represents the **overall conversion rate across both groups combined**, assuming the null hypothesis is true.

In [16]:
p_pooled = (x_control + x_exposed) / (n_control + n_exposed)
p_pooled

0.46017699115044247

In [ ]:
p_pooled = (x_control + x_exposed) / (n_control + n_exposed)
p_pooled

### Standard error tells us:

how much the difference in conversion rates would vary just due to randomness.

In [20]:
SE = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_control + 1/n_exposed))
SE 

0.028319932727228023

### Z-Score (THIS is your test statistic)

Z-score tells us:

how many standard errors away from 0 our observed difference is

In [21]:
z = (p_exposed - p_control)/SE 
z 

0.645680181221037

### Interpretation

Z near 0 → difference is tiny vs noise

|Z| > ~2 → difference is large vs noise

### The p-value is:

If H₀ is true, what is the probability of seeing a Z-score at least this extreme?

In [22]:
p_value = 2 * norm.sf(abs(z))
p_value

0.5184864982198968

Interpretation

If p < 0.05 → unlikely under null → reject H₀

If p ≥ 0.05 → plausible under null → fail to reject H₀

CI gives a range of plausible true effects.

In [23]:
z_crit = norm.ppf(1 - 0.05/2)
ci_low = diff - z_crit * SE
ci_high = diff + z_crit * SE
(z_crit, ci_low, ci_high)

(1.959963984540054, -0.03722042889447995, 0.07379166748544828)

Interpretation

If CI contains 0 → not significant

If CI excludes 0 → significant

alpha = 0.05
decision = "Reject H0" if p_value < alpha else "Fail to Reject H0"
decision

We fail to reject H0: we do not have strong evidence that the new ad improves YES responses.